In [1]:
import os 
os.chdir('../')
%pwd

'c:\\Users\\DELL\\Desktop\\drink_quality_prediction'

In [2]:
from pathlib import Path
from dataclasses import dataclass
from typing import Dict


@dataclass(frozen=True)
class ModelTrainerConfig:
    
    root_dir: Path
    
    X_train_path: Path
    X_test_path: Path
    
    y_train_path: Path
    y_test_path: Path
    
    model_name: str
    
    params: Dict

In [3]:
# configManager:
from mlproject.utils.common import read_yaml, create_directories
from mlproject.constants import * 


In [4]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        
        config = self.config.model_trainer
        params = self.params.random_forest
        
        create_directories([config.root_dir])
        
        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            X_train_path=Path(config.X_train_path),
            X_test_path=Path(config.X_test_path),
            y_train_path=Path(config.y_train_path),
            y_test_path=Path(config.y_test_path),
            model_name=config.model_name,
            params=params
        )
        
        return model_trainer_config

In [5]:
import pandas as pd 
import os 
from mlproject import logger 
from sklearn.ensemble import RandomForestClassifier 
import joblib 


In [6]:
import os
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report


class ModelTrainer:
    
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        
        # ============================
        # 1️⃣ Load Data
        # ============================
        
        X_train = pd.read_csv(self.config.X_train_path)
        X_test = pd.read_csv(self.config.X_test_path)
        y_train = pd.read_csv(self.config.y_train_path).values.ravel()
        y_test = pd.read_csv(self.config.y_test_path).values.ravel()

        # ============================
        # 2️⃣ Initialize Model with YAML Params
        # ============================
        
        rf = RandomForestClassifier(**self.config.params)

        # ============================
        # 3️⃣ Train Model
        # ============================
        
        rf.fit(X_train, y_train)

        # ============================
        # 4️⃣ Evaluate
        # ============================
        
        y_pred = rf.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        print("Accuracy:", acc)
        print("F1 Score:", f1)
        print("\nClassification Report:\n")
        print(classification_report(y_test, y_pred))

        # ============================
        # 5️⃣ Save Model
        # ============================
        
        model_path = os.path.join(self.config.root_dir, self.config.model_name)
        joblib.dump(rf, model_path)

        print(f"\nModel saved at: {model_path}")

        return acc, f1

In [8]:

try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
    
except Exception as e:
    raise e

[2026-03-16 20:11:47,616] INFO: common: yaml file: config\config.yaml loaded successfully
[2026-03-16 20:11:47,619] INFO: common: yaml file: params.yaml loaded successfully
[2026-03-16 20:11:47,627] INFO: common: yaml file: schema.yaml loaded successfully
[2026-03-16 20:11:47,627] INFO: common: created directory at: artifacts
[2026-03-16 20:11:47,631] INFO: common: created directory at: artifacts/model_trainer
Accuracy: 0.7720588235294118
F1 Score: 0.7680798004987531

Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.84      0.78       192
           1       0.83      0.71      0.77       216

    accuracy                           0.77       408
   macro avg       0.78      0.78      0.77       408
weighted avg       0.78      0.77      0.77       408


Model saved at: artifacts\model_trainer\model.joblib
